# Hypothesis H3: Orders with More Than 30% Discount Have Significantly Lower Profit Margins

This notebook evaluates whether higher discounting is associated with lower profit margins.

## Business objective
Measure the relationship between discount percentage and profit margin and assess whether orders with discounts above 30% are materially less profitable.

## Hypotheses
- **H0:** There is no significant relationship between discount percentage and profit margin.
- **H1:** Orders with discounts greater than 30% have significantly lower profit margins.

## Statistical approach
- Threshold check for orders with `discount_pct > 30`
- Profit margin summary statistics
- Correlation analysis using Pearson and Spearman
- Scatter plot with regression trend line

## Dataset used
- Source requested: `orders_cleaned.csv`
- Source available in this project: `data/cleaned/orders_clean.csv`

**Important note:** this notebook explicitly checks whether the dataset contains any orders above the 30% threshold before trying to test the hypothesis directly.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'cleaned' / 'orders_clean.csv'
OUTPUT_DIR = PROJECT_ROOT / 'Reports' / 'h3_discount_profitability'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH, OUTPUT_DIR

In [ ]:
orders = pd.read_csv(DATA_PATH)
orders.head()

## Prepare the analysis dataset

In [ ]:
analysis_df = orders[['order_id', 'order_date', 'discount_pct', 'profit_margin_pct', 'order_value_usd', 'gross_margin_usd', 'order_status']].copy()
analysis_df['order_date'] = pd.to_datetime(analysis_df['order_date'], errors='coerce')
analysis_df['discount_pct'] = pd.to_numeric(analysis_df['discount_pct'], errors='coerce')
analysis_df['profit_margin_pct'] = pd.to_numeric(analysis_df['profit_margin_pct'], errors='coerce')
analysis_df = analysis_df.dropna(subset=['discount_pct', 'profit_margin_pct'])
analysis_df = analysis_df[(analysis_df['discount_pct'] >= 0) & (analysis_df['profit_margin_pct'].between(-100, 100))].copy()
analysis_df['high_discount_flag'] = analysis_df['discount_pct'] > 30
analysis_df.head()

## Threshold check for discounts above 30%

In [ ]:
threshold_check = pd.DataFrame([
    {
        'total_orders': len(analysis_df),
        'orders_above_30pct_discount': int(analysis_df['high_discount_flag'].sum()),
        'max_discount_pct': analysis_df['discount_pct'].max(),
        'min_discount_pct': analysis_df['discount_pct'].min()
    }
])

threshold_check

In [ ]:
threshold_check.to_csv(OUTPUT_DIR / 'discount_threshold_check.csv', index=False)
print(f'Saved threshold check to: {OUTPUT_DIR / "discount_threshold_check.csv"}')

## Profit margin statistics
This compares overall order profitability and, when available, the `>30%` subset.

In [ ]:
profit_margin_summary = (
    analysis_df
    .groupby('high_discount_flag')['profit_margin_pct']
    .agg(
        order_count='size',
        mean_profit_margin='mean',
        median_profit_margin='median',
        std_profit_margin='std',
        min_profit_margin='min',
        max_profit_margin='max'
    )
)

if profit_margin_summary.empty:
    profit_margin_summary = pd.DataFrame()

profit_margin_summary

In [ ]:
overall_profit_margin_summary = analysis_df['profit_margin_pct'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).to_frame(name='overall')
overall_profit_margin_summary

## Correlation method selection
Shapiro-Wilk tests are used as a distribution diagnostic. If the variables are not close to normal, Spearman is the safer primary measure.

In [ ]:
distribution_checks = []
for column in ['discount_pct', 'profit_margin_pct']:
    sample_for_test = analysis_df[column].sample(5000, random_state=42) if len(analysis_df) > 5000 else analysis_df[column]
    stat, p_value = stats.shapiro(sample_for_test)
    distribution_checks.append({
        'variable': column,
        'n_used_for_test': len(sample_for_test),
        'shapiro_statistic': stat,
        'shapiro_p_value': p_value
    })

distribution_checks = pd.DataFrame(distribution_checks)
distribution_checks

In [ ]:
primary_method = 'Spearman' if (distribution_checks['shapiro_p_value'] < 0.05).any() else 'Pearson'
primary_method

## Correlation analysis

In [ ]:
correlation_matrix = analysis_df[['discount_pct', 'profit_margin_pct']].corr(method='spearman' if primary_method == 'Spearman' else 'pearson')
correlation_matrix

In [ ]:
pearson_stat, pearson_p = stats.pearsonr(analysis_df['discount_pct'], analysis_df['profit_margin_pct'])
spearman_stat, spearman_p = stats.spearmanr(analysis_df['discount_pct'], analysis_df['profit_margin_pct'])

correlation_test_results = pd.DataFrame([
    {
        'method': 'Pearson',
        'correlation_coefficient': pearson_stat,
        'p_value': pearson_p,
        'significant_at_0_05': pearson_p < 0.05
    },
    {
        'method': 'Spearman',
        'correlation_coefficient': spearman_stat,
        'p_value': spearman_p,
        'significant_at_0_05': spearman_p < 0.05
    }
])

correlation_test_results

In [ ]:
correlation_matrix.to_csv(OUTPUT_DIR / 'correlation_matrix.csv')
correlation_test_results.to_csv(OUTPUT_DIR / 'correlation_test_results.csv', index=False)
distribution_checks.to_csv(OUTPUT_DIR / 'distribution_checks.csv', index=False)
profit_margin_summary.to_csv(OUTPUT_DIR / 'profit_margin_summary_by_threshold.csv')
overall_profit_margin_summary.to_csv(OUTPUT_DIR / 'overall_profit_margin_summary.csv')
print('Saved correlation and summary tables.')

## Scatter plot and regression trend line
The plot uses a random sample for readability while the statistics above use the full dataset.

In [ ]:
plot_sample = analysis_df.sample(min(12000, len(analysis_df)), random_state=42)

fig, ax = plt.subplots(figsize=(12, 7))
sns.regplot(
    data=plot_sample,
    x='discount_pct',
    y='profit_margin_pct',
    scatter_kws={'alpha': 0.18, 's': 18, 'color': '#1565C0'},
    line_kws={'color': '#C62828', 'linewidth': 2.5},
    ax=ax
)
ax.set_title('Discount Percentage vs Profit Margin Percentage')
ax.set_xlabel('Discount Percentage')
ax.set_ylabel('Profit Margin Percentage')
plt.tight_layout()
plot_path = OUTPUT_DIR / 'scatter_regression_discount_vs_profit_margin.png'
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved scatter plot to: {plot_path}')

## Statistical interpretation and pricing recommendation

In [ ]:
orders_above_30 = int(threshold_check.loc[0, 'orders_above_30pct_discount'])
max_discount = float(threshold_check.loc[0, 'max_discount_pct'])

if primary_method == 'Spearman':
    primary_corr = spearman_stat
    primary_p = spearman_p
else:
    primary_corr = pearson_stat
    primary_p = pearson_p

if orders_above_30 == 0:
    hypothesis_conclusion = (
        'The >30% discount hypothesis cannot be directly tested because the cleaned dataset contains no orders above 30% discount.'
    )
else:
    if primary_p < 0.05 and primary_corr < 0:
        hypothesis_conclusion = 'The data supports the claim that heavier discounting is associated with lower profit margins.'
    elif primary_p < 0.05 and primary_corr > 0:
        hypothesis_conclusion = 'The relationship is statistically significant but positive, not negative.'
    else:
        hypothesis_conclusion = 'The analysis does not find a statistically significant relationship.'

if primary_p < 0.05 and primary_corr > 0:
    pricing_recommendation = (
        'Do not assume larger discounts automatically erode margin in this dataset. Review pricing rules, product mix, and margin calculation logic before tightening discount policies.'
    )
elif primary_p < 0.05 and primary_corr < 0:
    pricing_recommendation = (
        'Set tighter approval thresholds for larger discounts and monitor whether incremental volume offsets the lower margin rate.'
    )
else:
    pricing_recommendation = (
        'Keep discount approvals tied to segment, deal size, and product economics rather than relying on discount percentage alone.'
    )

report_lines = [
    '# Pricing Recommendation Report: Hypothesis H3',
    '',
    '## Executive summary',
    hypothesis_conclusion,
    '',
    '## Data coverage check',
    f'- Total analyzable orders: {len(analysis_df):,}',
    f'- Orders above 30% discount: {orders_above_30:,}',
    f'- Maximum observed discount: {max_discount:.2f}%',
    '',
    '## Correlation findings',
    f'- Primary method used: {primary_method}',
    f'- Pearson correlation: {pearson_stat:.4f} (p-value={pearson_p:.4e})',
    f'- Spearman correlation: {spearman_stat:.4f} (p-value={spearman_p:.4e})',
    '',
    '## Statistical interpretation',
    '- The normality diagnostics are not supportive of a purely parametric assumption, so Spearman is treated as the primary relationship measure.',
    '- In this dataset, discount percentage and profit margin move in the same direction, which is the opposite of the original hypothesis.',
    '- Because the thresholded >30% group does not exist in the data, any threshold-specific conclusion would require additional records.',
    '',
    '## Pricing recommendation',
    f'- {pricing_recommendation}',
    '- Confirm whether `profit_margin_pct` is defined after discounts and rebates, not before them.',
    '- If management wants a true >30% discount test, extend the dataset or source historical campaigns where such discounts actually occurred.'
]

report_text = '
'.join(report_lines)
report_path = OUTPUT_DIR / 'pricing_recommendation_report.md'
report_path.write_text(report_text, encoding='utf-8')

print(report_text)
print(f'
Saved report to: {report_path}')

## Conclusion
This notebook preserves the original business question, checks whether the threshold exists in the data, and still delivers a usable statistical readout for discount strategy decisions.